<left>
<table style='margin-top:0px; margin-left:0px;'>
<tr>
  <td><img src='https://raw.githubusercontent.com/worm-portal/WORM-Figures/master/style/worm.png' alt='WORM' title='WORM' width=50/></td>
  <td><h1 style=font-size:30px>Media Maker</h1><h2>Notebook</h2></td>
</tr>
</table>
</left>

This notebook provides a workflow for designing or evaluating microbial growth media based on environmental geochemistry. It supports two primary use cases: (1) generating a new media recipe from geochemical data and a defined reagent list, and (2) loading an existing recipe to calculate resulting ion concentrations for further analysis (e.g., aqueous speciation or energy supply calculations). The conceptual framework, validation, and example applications of this tool are described in detail in <i>Smania (2026)</i> (full thesis available via ProQuest).

In [1]:
# tools
from tools import *

reset: thermodynamic system initialized
The WORM thermodynamic database has been loaded: 1713 aqueous, 2000 total species


### Inputting a Geochemical Data File 

Geochemical input data are typically derived from ion chromatography (IC) or inductively coupled plasma (ICP/ICP-MS) analyses. The input file must follow the structure defined in speciation-formatting.csv, including consistent headers and unit formatting. Users may either populate the provided template or reformat their dataset to match it. The sample identifier used throughout the workflow is automatically read from the Name column in the input file.

In [2]:
sample = "Perpetual Spouter"
input_file="example-input.csv"

A predefined list of excluded ions is included, currently consisting of dissolved gases and selected toxic or non-relevant species. This list can be modified depending on the goals of the experiment. Users should exclude ions that are not biologically relevant, not supported by the reagent database, or not intended to be reproduced in the media formulation.

In [3]:
#exclude any unwanted ions
excluded = {'CH4','CO','H2','O2','CO2','N2','Ar', #gases
            'Tl+','Be+2','CrO4-2',  #highly toxic
            'Th+4','U+4', #radioactive
            'NH3', 'Ga+3','Y+3','Zr+4','Sn+2','Cs+','La+3','Li+','Hf+4','NbO3-','SbO2-', #not available in master-reagents
            }

This step converts the raw geochemical dataset into a format compatible with the optimization and downstream calculations. It applies ion exclusions and standardizes the structure for internal use.

An optional charge balancing step can be applied to a selected ion. Charge balancing is recommended because laboratory reagents are introduced as electrically neutral compounds (cation–anion pairs), meaning the resulting media will inherently be charge balanced. Ensuring the input data are also charge balanced improves the optimizer’s ability to accurately reproduce target ion concentrations.

In [4]:
formatted_df, geochem_dict = format_input(input_file, sample,
    excluded_target_ions=excluded,
    charge_balance=True,
    charge_balance_ion="Cl-"                                      
)

Loading Water-Organic-Rock-Microbe (WORM) thermodynamic databases...
wrm_data_latest.csv is now set as the active thermodynamic database.
Element database elements.csv is active.
Solid solution database solid_solutions.csv is active.
LogK database wrm_data_logK.csv is active.
LogK_S database wrm_data_logK_S.csv is active.
Loading thermodynamic database into pyCHNOSZ...

Balanced dissociation reactions are missing for 2 species:
  dimethylamine, dimethylaminium

Generating dissociation reactions for these species using strict and auxiliary basis species containing a maximum of one atom of one element besides O and H...

dimethylamine : -1.0000 dimethylamine 2.0000 HCO3- 1.0000 NH4+ 1.0000 H+ -3.0000 O2(g)
dimethylaminium : -1.0000 dimethylaminium 2.0000 HCO3- 1.0000 NH4+ 2.0000 H+ -3.0000 O2(g)
A column to set sample redox state could not be automatically detected in the input file. Valid columns include 'logfO2' (with subheader 'logfO2'), 'O2' (with a valid concentration subheader), 'E

### Gathering reagents!

The default reagent database (master-reagents.csv) includes 73 commonly used laboratory reagents along with associated ion contributions and metadata (e.g., autoclavability). Users may substitute a custom reagent file if desired. If a custom file is used, it must define ion stoichiometry, and any missing metadata (such as autoclavability) will not be reflected in the generated recipe output. Reagent-level exclusions can also be applied to restrict which compounds are available during optimization.

In [5]:
excluded_reagents = {}

#load default reagents
reagents = load_reagents(excluded_reagents=excluded_reagents)
#load your own list of reagents
reagents_csv_path="master-reagents.csv"
reagents = load_reagents(reagents_csv_path, excluded_reagents=excluded_reagents)

Loaded reagent definitions from default CSV: /var/lib/private/sisalt/Media-Tools/tools/master-reagents.csv
✅ Reagents loaded: 73 entries.
Loaded reagent definitions from your CSV: master-reagents.csv
✅ Reagents loaded: 73 entries.


This step verifies that all ions present in the geochemical dataset can be supplied by at least one reagent in the database. Any ions not represented in the reagent list will be flagged. These gaps must be resolved before proceeding by either adding appropriate reagents to the database or excluding the ion from the dataset if it is not required for the intended application.

In [6]:
# check that you have sufficient reagents to cover the ions in your data
check_ion_coverage(reagents, geochem_dict)

✅ All target ions covered by reagent set.


### Calculating the recipe!

This step performs an optimization to determine the combination of reagents that best reproduces the target ion concentrations from the input geochemical dataset. Using the stoichiometry defined in the reagent file, the code solves for reagent amounts that collectively match the desired ion composition while minimizing unintended contributions.

To improve accuracy and practical usability, the workflow separates the calculation into two components: base ions and trace ions. Base ions (major constituents) are solved together because they dominate solution chemistry and are typically added in larger, directly measurable amounts. Trace ions are calculated separately to prevent them from being overwhelmed or excluded during optimization due to their low concentrations. This separation ensures that trace elements are retained and can be added through concentrated stock solutions without significantly altering the bulk chemistry of the medium.


In [7]:
base_df, trace_stock_df, ions_total_df, meta = create_recipe(reagents, geochem_dict,
    trace_threshold=1e-6,
    trace_ions=[],
    trace_stock_mL_per_L=1.0,
)

# combine ions in trace stock and base stock
all_ions = meta["all_ions"]
ion_contributions_total = meta["ions_total_final"]
export_final_ion_concentrations(all_ions, ion_contributions_total)

✅ Final ion concentrations saved to: final_ion_concentrations.csv


PosixPath('final_ion_concentrations.csv')

### Recipe Organization and Stock Preparation

Once reagent concentrations are calculated, the recipe is reorganized into a format suitable for laboratory preparation. Reagents are filtered to ensure they meet practical handling thresholds: each must correspond to at least 0.01 g in a stock solution and 0.01 mL when pipetted into the final medium. Reagents below these thresholds are grouped into a serial dilution stock.

Reagents are grouped based on:

* Mass scale (to keep stock concentrations practical)
* Autoclavability and compatibility (if metadata are available in the reagent file)

All stock solutions are defined relative to preparation of 1 L of final media, providing a consistent reference scale.



For trace elements, a serial dilution approach is used to ensure accurate and measurable additions:

1. A highly concentrated primary stock is prepared containing all trace reagents.
2. A working stock is created by diluting a small volume of the primary stock (commonly 1 mL into 1 L).
3. A fixed volume of the working stock is then added to the final medium.

This approach allows very small target concentrations to be achieved reproducibly without requiring impractically small measurements.

In [8]:
combined_df = merge_trace_into_base(base_df, trace_stock_df, trace_stock_mL_per_L=1.0)

print_organized_recipe(combined_df, reagents_csv_path,
                                            stock_volume_mL=1000, serial_dilutions=True)


== USER-FRIENDLY PREP ==


=== BASE STOCKS ===

Group I — 10000× base stock (weigh into 1000 mL, add 0.1 mL per L):
  9.266 g NaCl
  0.935 g KCl
  0.692 g H3BO3
  0.316 g CaCl2
  0.174 g NaF
  0.072 g KH2AsO4
  0.032 g KBr

Group II — 100000× base stock (weigh into 1000 mL, add 0.01 mL per L):
  0.060 g MgCl2•6H2O
  0.059 g Al2(SO4)3
  0.017 g Na2MoO4•2H2O
  0.012 g MnCl2•4H2O

Group III — NON-AUTOCLAVABLE — 10000× base stock (weigh into 1000 mL, add 0.1 mL per L):
  2.622 g Na2SiO3
  0.021 g KHCO3

Group IV — NON-AUTOCLAVABLE — 100000× base stock (weigh into 1000 mL, add 0.01 mL per L):
  0.071 g RbCl
  0.025 g FeSO4•7H2O
  0.023 g Na2S•9H2O
  0.023 g H2WO4


=== TRACE SERIAL STOCKS ===

Group I — Compatible trace stock (serial dilution)
  Step 1 — Primary stock (100000000×, 1 L):
    Weigh into 1 L DI water (target: <= 10 g max reagent):
      6.325 g SrCl2
      2.078 g BaCl2
      1.274 g ZnSO4•7H2O

  Step 2 — Working stock (100000×, 1 L):
    Add 1 mL primary stock to 1 L DI wat

### How similar is your recipe to your source?

This step evaluates how well the generated recipe reproduces the original geochemical dataset.

A weighted similarity score is calculated to quantify overall agreement between the target and recipe ion concentrations. In general form:

$$
\text{Similarity} = 1 - \frac{\sum_i w_i |C_{i,\text{recipe}} - C_{i,\text{target}}|}{\sum_i w_i C_{i,\text{target}}}
$$

where:

* C<sub>i,target</sub> is the target concentration of ion i
* C<sub>i,recipe</sub> is the concentration produced by the recipe
* w<sub>i</sub> is a weighting factor (often used to emphasize trace vs. major ions)

In addition to the overall score, the code calculates percent difference for each ion:

$$
\% \text{Difference}_i = \left| \frac{C_{i,\text{recipe}} - C_{i,\text{target}}}{C_{i,\text{target}}} \right| \times 100
$$

These per-ion values are useful for identifying which species contribute most to any mismatch between the recipe and the source data.

In [9]:
# Calculating the overall similarity to the geochemical composition
similarity = calculate_similarity(all_ions, ion_contributions_total, geochem_dict)
diff = percent_diff_per_ion(all_ions, ion_contributions_total, geochem_dict)


Overall similarity to original target: 92.547465%

== Percent Difference per Ion (Readable) ==

≈0% diff (|%diff| < 0.0001%):
HCO3-, F-, Br-, K+, B(OH)3, Al+3, VO+2, Mn+2, Co+2, Ni+2, Cu+2, Zn+2, H2AsO4-, Rb+, Sr+2, MoO4-2, Cd+2, Ba+2, WO4-2, HS-, Fe+2, HPO4-2, Mg+2, Ca+2

Missing from solution (target > 0, actual ~ 0):
Pb+2, Fe+3

Other ions:
SO4-2: -98.474%
SiO2: -59.223%
Cl-: +0.012%
Na+: +0.011%


### Format your file for speciation

This step converts the generated recipe into a format compatible with the speciation notebook. The output is structured to match the required input format for downstream speciation calculations.

The user is prompted to define the temperature and pH (H⁺) for the recipe. These values are appended to the dataset, and a formatted file is generated that can be directly used in the speciation notebook without additional preprocessing.

In [10]:
# Make sure to change the sample and csv file name to your own data
formatted_file = build_speciation_recipe_file(sample,
    op_path=input_file,
    final_path="final_ion_concentrations.csv",
    fmt_path="speciation-formatting.csv" #don't change this
)

Enter RECIPE pH (to store in H+ column):  7.3
Enter RECIPE Temperature (°C):  88



Appending new ion columns:
 - HPO4-2

Wrote: formatted__Perpetual Spouter.csv


### Want to get ion concentrations from an existing media recipe?

You can start here!

In [1]:
#load in our tools
from tools import *

reset: thermodynamic system initialized
The WORM thermodynamic database has been loaded: 1713 aqueous, 2000 total species


This step takes an existing media recipe (defined by reagent concentrations) and converts it into its corresponding ion composition. Accepted reagent units include: g/L, mg/L, or mol/L but they all must be in the same unit and the unit must be notated in the row below the header for 'concentration'. <i>SFCM.csv</i> serves as an example for how the input file should be formatted.

Using the stoichiometry defined in the reagent database, the function calculates how much of each ion is contributed by each reagent and compiles the total ion concentrations in solution.

The output is a formatted file (<i>recipe_ion_molality.csv</i>) that reports ion concentrations as molality and includes the specified environmental conditions (pH, temperature, and pressure). This file can be used directly for downstream analyses such as aqueous speciation or energy supply calculations.

In [2]:
convert_reagents_to_ions(
    reagent_csv_path="SFCM.csv",
    output_csv_path="recipe_ion_molality.csv",
    #master_reagents_path="master-reagents.csv",
    pH=7.5,
    temperature=72,
    pressure=1
)

,Sample,Name,Pressure,Year,H+,Temperature,O2,HCO3-,Cl-,F-,...,Fe+2,Fe+3,PO4-3,Li+,Mg+2,Ca+2,NH4+,NO2-,NO3-,EDTA-4
0,id,name,bar,year,pH,degC,ppm,Molality,Molality,Molality,...,Molality,Molality,Molality,Molality,Molality,Molality,Molality,Molality,Molality,Molality
1,recipe,SFCM,1,,7.5,72,,0.004,0.525254,,...,,0.000008,0.000294,,0.024594,0.013514,0.001,,,0.000008


### Want to adjust any ions?

This function allows you to adjust individual ion concentrations in either your generated recipe or your pre-existing recipe inputted above. This can then be turned back into an optimized recipe using a chosen set of reagents.

The function will keep asking if you want to change any ion concentrations until the user types 'n'.

In [3]:
final_total_concentrations = adjust_ion_concentrations(
    input_csv="final_ion_concentrations.csv", #or recipe_ion_molality.csv for adjusting recipe concentrations
    output_csv="adjusted_ion_concentrations.csv"
)

Do you want to change the concentrations of any ions? (y/n):  n


No changes made.


### Get a new recipe with your new concentrations

These next few blocks work about the same as the previous ones, but allow you to also input pre-existing recipe data.

Tasks that can be complete using this section:
* Regenerating a recipe after adjusting ion concentrations in your <b>generated recipe</b>
* Regenerating a recipe after adjusting ion concentrations in a <b>pre-existing recipe</b>
* Changing a pre-existing recipe to match reagents you have access to in your lab

This first function loads in your chosen reagents. This can either be the default <i>master-reagents.csv</i> (type="custom") or a custom set of reagents of the same formatting (type="custom")

It is also possible to select type="preexist". This will take the reagents in the inputted pre-existing recipe file and cross-match them with the default database. If some reagents aren't in the database, they will be listed. 

In [7]:
reagents_csv_path="master-reagents.csv"
reagents = load_reagents(reagents_csv_path)

#select which reagents you want to use
selected_reagents = select_reagents(
    type="custom", # or "preexist"
    reagents=reagents
)

Loaded reagent definitions from your CSV: master-reagents.csv
✅ Reagents loaded: 73 entries.

Reagent source: previously loaded reagents (custom recipe workflow)
Number of reagents available: 73


In [8]:
#reassign the input file and sample if needed
input_file = "recipe_ion_molality.csv"
sample = "SFCM"

#format your file
formatted_df, geochem_dict = format_input(input_file, sample)


✅ Formatted DataFrame (after exclusions):
       ion  concentration
0    HCO3-       0.004000
1      Cl-       0.525254
2      Br-       0.000840
3    SO4-2       0.000501
4      Na+       0.449511
5       K+       0.001134
6     Mn+2       0.000505
7     Co+2       0.000799
8     Ni+2       0.000101
9     Cu+2       0.000012
10    Zn+2       0.000501
11  MoO4-2       0.000149
12    Fe+3       0.000008
13   PO4-3       0.000294
14    Mg+2       0.024594
15    Ca+2       0.013514
16    NH4+       0.001000
17  EDTA-4       0.000008


This section will ultimately do the same exact thing as the optimizer in the beginning of the notebook, but it is condensed and accesible in this section of the notebook for your convenience.

If you're new here, reagents are grouped based on:

* Mass scale (to keep stock concentrations practical)
* Autoclavability and compatibility (if metadata are available in the reagent file)

All stock solutions are defined relative to preparation of 1 L of final media, providing a consistent reference scale.

For trace elements, a serial dilution approach is used to ensure accurate and measurable additions:

1. A highly concentrated primary stock is prepared containing all trace reagents.
2. A working stock is created by diluting a small volume of the primary stock (commonly 1 mL into 1 L).
3. A fixed volume of the working stock is then added to the final medium.

This approach allows very small target concentrations to be achieved reproducibly without requiring impractically small measurements.

In [9]:
#reconstruct the recipe
base_df, trace_stock_df, ions_total_df, meta = create_recipe(
    selected_reagents,
    geochem_dict,
    trace_threshold=1e-6,
    trace_ions=[],
    trace_stock_mL_per_L=1.0,
)

# combine ions in trace stock and base stock
all_ions = meta["all_ions"]
ion_contributions_total = meta["ions_total_final"]
export_final_ion_concentrations(all_ions, ion_contributions_total)
combined_df = merge_trace_into_base(base_df, trace_stock_df, trace_stock_mL_per_L=1.0)

#print a user-friendly recipe
print_organized_recipe(combined_df, reagents_csv_path, stock_volume_mL=1000, serial_dilutions=True)

✅ Final ion concentrations saved to: final_ion_concentrations.csv

== USER-FRIENDLY PREP ==


=== BASE STOCKS ===

Group I — 100× base stock (weigh into 1000 mL, add 10 mL per L):
  2.601 g NaCl
  0.500 g MgCl2•6H2O
  0.149 g CaCl2
  0.024 g NaHCO3
  0.019 g CoCl2•6H2O

Group II — 100000× base stock (weigh into 1000 mL, add 0.01 mL per L):
  10.000 g MnCl2•4H2O
  7.112 g Na2SO4
  3.600 g Na2MoO4•2H2O
  3.380 g NH4H2PO4
  2.400 g NiCl2•6H2O
  0.280 g FeNaEDTA
  0.200 g CuCl2•2H2O

Group III — 10000× base stock (weigh into 1000 mL, add 0.1 mL per L):
  0.134 g CaBr2

Group IV — NON-AUTOCLAVABLE — 10000× base stock (weigh into 1000 mL, add 0.1 mL per L):
  1.135 g KHCO3
  0.691 g NH4Br
  0.683 g ZnCl2

